# In this notebook we are going to cover some the most important fundamental concepts of Tensors using TensorFlow.

More specifically , we are going to cover:
* Introduction to Tensors
* Getting information from Tensors
* Manipulating Tensors
* Tensors & NumPy
* Using @tf.function (a way to speed up your regular Python function)
* Using GPUs with TensorFlow (or TPUs)
* Exercises

# Introduction to Tensors

In [3]:
# Import TensorFlow
import tensorflow as tf
print(tf.__version__)

2.19.0


## tf.constant()

In [4]:
# Create Tensors with tf.constant()

scaler = tf.constant(7)
scaler

<tf.Tensor: shape=(), dtype=int32, numpy=7>

In [5]:
# Check the numner of dimensions of Tensor (ndim stands for number of dimensions)
scaler.ndim

0

In [6]:
# Create a vector
vector = tf.constant([10,10])
vector

<tf.Tensor: shape=(2,), dtype=int32, numpy=array([10, 10], dtype=int32)>

In [7]:
# Check dimensions of vector
vector.ndim

1

In [8]:
# Create a matrix (which has more than 1 dimensions)
matrix = tf.constant([[1,2,3],[4,5,6],[7,8,9]])
matrix

<tf.Tensor: shape=(3, 3), dtype=int32, numpy=
array([[1, 2, 3],
       [4, 5, 6],
       [7, 8, 9]], dtype=int32)>

In [9]:
matrix.ndim

2

In [10]:
# Create another matrix apply parameters
another_matrix = tf.constant([[10.,7.],[3.,2.],[8.,9.]],dtype=tf.float16)
another_matrix

<tf.Tensor: shape=(3, 2), dtype=float16, numpy=
array([[10.,  7.],
       [ 3.,  2.],
       [ 8.,  9.]], dtype=float16)>

In [11]:
another_matrix.ndim

2

In [12]:
# Create Tensor
tensor = tf.constant([[[1,2,3],
                       [4,5,6]],
                      [[7,8,9],
                       [10,11,12]],
                     [[13,14,15],
                      [16,17,18]]])
tensor

<tf.Tensor: shape=(3, 2, 3), dtype=int32, numpy=
array([[[ 1,  2,  3],
        [ 4,  5,  6]],

       [[ 7,  8,  9],
        [10, 11, 12]],

       [[13, 14, 15],
        [16, 17, 18]]], dtype=int32)>

In [13]:
tensor.ndim

3

Definations

* Scaler: a single number
* Vector: a number with direction (eg, wind speed and direction)
* Matrix: a 2-dimensional array of numbers
* Tensor: an n-dimensional array of numbers (when n can be any number a 0-D tensor is a scaler, a 1-D tensor is a vecotr and 2-D tensor is an matrix)

## tf.Variable

In [14]:
# Create the same tensor using tf.Variable() as above
changeable_tensor = tf.Variable([10,7])
unchangeable_tensor = tf.constant([10,7])
changeable_tensor, unchangeable_tensor


(<tf.Variable 'Variable:0' shape=(2,) dtype=int32, numpy=array([10,  7], dtype=int32)>,
 <tf.Tensor: shape=(2,), dtype=int32, numpy=array([10,  7], dtype=int32)>)

In [15]:
# Change one of the elements
changeable_tensor[0] = 7
changeable_tensor

TypeError: 'ResourceVariable' object does not support item assignment

In [16]:
# Use .assign() to change the elements value
changeable_tensor[0].assign(7)
changeable_tensor

<tf.Variable 'Variable:0' shape=(2,) dtype=int32, numpy=array([7, 7], dtype=int32)>

In [17]:
unchangeable_tensor[0].assign(7)
unchangeable_tensor

AttributeError: 'tensorflow.python.framework.ops.EagerTensor' object has no attribute 'assign'

**Note**:  Which one should you use? tf.constant() or tf.Variable()?

It will depend on what your problem requires. However, most of the time, TensorFlow will automatically choose for you (when loading data or modelling data).

## Create Random Tensors


### Random tensors are tensors of some abitrary size which contain random numbers.

### In Neural Networks initially the hidden layer or layer after intput weights are initialized with random.

In [18]:
# Create 2 random (but the same) tensors
random_1 = tf.random.Generator.from_seed(42) # set seed for reproducibility
random_1 = random_1.normal(shape=(3,2))
random_1

<tf.Tensor: shape=(3, 2), dtype=float32, numpy=
array([[-0.7565803 , -0.06854702],
       [ 0.07595026, -1.2573844 ],
       [-0.23193763, -1.8107855 ]], dtype=float32)>

In [19]:
random_2 = tf.random.Generator.from_seed(42)
random_2 = random_2.normal(shape=(3,2))   #normal means normal distribution
random_2

<tf.Tensor: shape=(3, 2), dtype=float32, numpy=
array([[-0.7565803 , -0.06854702],
       [ 0.07595026, -1.2573844 ],
       [-0.23193763, -1.8107855 ]], dtype=float32)>

In [20]:
# Are they equal?
random_1 == random_2

<tf.Tensor: shape=(3, 2), dtype=bool, numpy=
array([[ True,  True],
       [ True,  True],
       [ True,  True]])>

### Shuffling the order of elements in Tensors




In [21]:
# Shuffling a tensor (valuable when you want to shuffle your data so the inherent order does not effect learning)
not_shuffled = tf.constant([[10,7],
                            [3,4],
                            [2,5]])
not_shuffled.ndim

2

In [22]:
not_shuffled

<tf.Tensor: shape=(3, 2), dtype=int32, numpy=
array([[10,  7],
       [ 3,  4],
       [ 2,  5]], dtype=int32)>

In [23]:
# Shuffle out not_shuffled tensor
tf.random.shuffle(not_shuffled, seed=42)  #randomly shuffle tensors with its 1st Dimensions

# Gets different output each time

<tf.Tensor: shape=(3, 2), dtype=int32, numpy=
array([[ 2,  5],
       [ 3,  4],
       [10,  7]], dtype=int32)>

In [24]:
tf.random.shuffle(not_shuffled)

<tf.Tensor: shape=(3, 2), dtype=int32, numpy=
array([[ 3,  4],
       [10,  7],
       [ 2,  5]], dtype=int32)>

It looks like if we want our shuffle tensors to be in the same order, we have got to use the global level random seed as well as the operation level random seed:

Rule: If both the global and the operation seed are set: Both seeds are used in conjunction to determine the random sequence.

In [25]:
tf.random.set_seed(42) # global level random seed
tf.random.shuffle(not_shuffled, seed=42) # operation level random seed

<tf.Tensor: shape=(3, 2), dtype=int32, numpy=
array([[10,  7],
       [ 3,  4],
       [ 2,  5]], dtype=int32)>

we need to set **global level random seed** because a neural network initializes itself with random patterns, you get different results every time you run those experiments. So to make reproducible experiments, you need to shuffle your data in similar order , initialize with a similar random pattern and run the experiment.

## Creating Tensors from NumPy Arrays

In [26]:
tf.ones([10,7])

<tf.Tensor: shape=(10, 7), dtype=float32, numpy=
array([[1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1.]], dtype=float32)>

In [27]:
tf.zeros(shape=(3,4))

<tf.Tensor: shape=(3, 4), dtype=float32, numpy=
array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]], dtype=float32)>

### Turn NumPy arrays into Tensors

The main difference between Numpy arrays and TensorFlow tensors is that tensors can be run on a GPU (much faster for numnerical computing).

In [28]:
from re import X
# You can also turn NumPy arrays into Tensors
import numpy as np
numpy_A = np.arange(1,25,dtype=np.int32) # create a NumPy array between 1 and 25
numpy_A

# X = tf.constant(some_matrix)  # Capital for matrix or tensor
# y = tf.constant(vector)   #Non-capital for vector

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24], dtype=int32)

In [29]:
A = tf.constant(numpy_A, shape=(2,3,4))
A

<tf.Tensor: shape=(2, 3, 4), dtype=int32, numpy=
array([[[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]],

       [[13, 14, 15, 16],
        [17, 18, 19, 20],
        [21, 22, 23, 24]]], dtype=int32)>

In [30]:
B = tf.constant(numpy_A)
B

<tf.Tensor: shape=(24,), dtype=int32, numpy=
array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24], dtype=int32)>

In [31]:
A,B

(<tf.Tensor: shape=(2, 3, 4), dtype=int32, numpy=
 array([[[ 1,  2,  3,  4],
         [ 5,  6,  7,  8],
         [ 9, 10, 11, 12]],
 
        [[13, 14, 15, 16],
         [17, 18, 19, 20],
         [21, 22, 23, 24]]], dtype=int32)>,
 <tf.Tensor: shape=(24,), dtype=int32, numpy=
 array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24], dtype=int32)>)

In [32]:
2*3*4

24

## Getting Information from Tensors (Tensor Attributes)

When dealing with tensors you need to be aware of the following attribute:
* Shape
* Axis or Dimension
* Rank
* Size

In [33]:
# Create a rank 4 tensor (4-D)
rank_4_tensor = tf.zeros(shape=[2,3,4,5])
rank_4_tensor

<tf.Tensor: shape=(2, 3, 4, 5), dtype=float32, numpy=
array([[[[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]]],


       [[[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]]]], dtype=float32)>

In [34]:
rank_4_tensor[0]

<tf.Tensor: shape=(3, 4, 5), dtype=float32, numpy=
array([[[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]]], dtype=float32)>

In [35]:
rank_4_tensor[0][0]

<tf.Tensor: shape=(4, 5), dtype=float32, numpy=
array([[0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]], dtype=float32)>

In [36]:
rank_4_tensor.shape, rank_4_tensor.ndim, tf.size(rank_4_tensor)

(TensorShape([2, 3, 4, 5]), 4, <tf.Tensor: shape=(), dtype=int32, numpy=120>)

In [37]:
2*3*4*5

120

In [38]:
# Get various attributes of our Tensor

print("Datatype of every element:", rank_4_tensor.dtype)
print("Number of dimensions (rank):", rank_4_tensor.ndim)
print("Shape of tensor:", rank_4_tensor.shape)
print("Elements along the 0 axis:", rank_4_tensor.shape[0])
print("Elements along the last axis:", rank_4_tensor.shape[-1])
print("Total number of elements in our tensor:", tf.size(rank_4_tensor))

Datatype of every element: <dtype: 'float32'>
Number of dimensions (rank): 4
Shape of tensor: (2, 3, 4, 5)
Elements along the 0 axis: 2
Elements along the last axis: 5
Total number of elements in our tensor: tf.Tensor(120, shape=(), dtype=int32)


In [39]:
print("Total number of elements in our tensor:", tf.size(rank_4_tensor).numpy())

Total number of elements in our tensor: 120


## Indexing and Expanding Tensors

Tensors can be indexed just like Pyhton lists

In [40]:
# Ex,
some_list = [1,2,3,4]
some_list[:2]

[1, 2]

In [41]:
# Get the first 2 elements of each dimensions
rank_4_tensor[:2,:2,:2,:2]

<tf.Tensor: shape=(2, 2, 2, 2), dtype=float32, numpy=
array([[[[0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.]]],


       [[[0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.]]]], dtype=float32)>

In [42]:
# get the first element fo=rom each dimension from each index except for the final one
rank_4_tensor[:1,:1,:1]

<tf.Tensor: shape=(1, 1, 1, 5), dtype=float32, numpy=array([[[[0., 0., 0., 0., 0.]]]], dtype=float32)>

In [43]:
rank_4_tensor[:1,:1,:1,:]

<tf.Tensor: shape=(1, 1, 1, 5), dtype=float32, numpy=array([[[[0., 0., 0., 0., 0.]]]], dtype=float32)>

In [44]:
rank_4_tensor.shape

TensorShape([2, 3, 4, 5])

In [45]:
rank_4_tensor[:1,:1,:,:1]

<tf.Tensor: shape=(1, 1, 4, 1), dtype=float32, numpy=
array([[[[0.],
         [0.],
         [0.],
         [0.]]]], dtype=float32)>

In [46]:
# Create a rank 2 tensor (2D)
rank_2_tensor = tf.constant([[10,7],
                            [3,4]])
rank_2_tensor.shape, rank_2_tensor.ndim

(TensorShape([2, 2]), 2)

In [47]:
# Get the last item of each of our rank 2 tensor
rank_2_tensor[:,1:2]

<tf.Tensor: shape=(2, 1), dtype=int32, numpy=
array([[7],
       [4]], dtype=int32)>

In [48]:
# Get the last item of each of our rank 2 tensor
rank_2_tensor[:,-1]

<tf.Tensor: shape=(2,), dtype=int32, numpy=array([7, 4], dtype=int32)>

### Adding extra Dimension to rank 2 tensor

In [49]:
rank_3_tensor = rank_2_tensor[...,tf.newaxis] #same as [:,;, tf.newaxis]
rank_3_tensor

<tf.Tensor: shape=(2, 2, 1), dtype=int32, numpy=
array([[[10],
        [ 7]],

       [[ 3],
        [ 4]]], dtype=int32)>

In [50]:
# alternative to tf.newaxis '
tf.expand_dims(rank_2_tensor, axis= -1) # -1 means expand the final axis

<tf.Tensor: shape=(2, 2, 1), dtype=int32, numpy=
array([[[10],
        [ 7]],

       [[ 3],
        [ 4]]], dtype=int32)>

In [51]:
tf.expand_dims(rank_2_tensor, axis= 0)

<tf.Tensor: shape=(1, 2, 2), dtype=int32, numpy=
array([[[10,  7],
        [ 3,  4]]], dtype=int32)>

In [52]:
tf.expand_dims(rank_2_tensor, axis= 1)

<tf.Tensor: shape=(2, 1, 2), dtype=int32, numpy=
array([[[10,  7]],

       [[ 3,  4]]], dtype=int32)>

## Manipulating Tensors with Basic Operations

### Baisc Operations
`+`,`-`,`*`,`/`

In [53]:
# You can add values to a tensor using the addition operator
tensor = tf.constant([[10,7],[3,4]])
tensor + 10

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[20, 17],
       [13, 14]], dtype=int32)>

In [54]:
# Original Tensor is unchanged
tensor

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[10,  7],
       [ 3,  4]], dtype=int32)>

In [55]:
tensor * 10

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[100,  70],
       [ 30,  40]], dtype=int32)>

In [56]:
tensor - 10

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[ 0, -3],
       [-7, -6]], dtype=int32)>

In [57]:
tensor / 10

<tf.Tensor: shape=(2, 2), dtype=float64, numpy=
array([[1. , 0.7],
       [0.3, 0.4]])>

In [58]:
# We can use the tensorflow build-in function
tf.multiply(tensor, 10)   # to have advantages of TensorFlow use this for fast execution on large data

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[100,  70],
       [ 30,  40]], dtype=int32)>

In [59]:
tf.add(tensor, 10)

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[20, 17],
       [13, 14]], dtype=int32)>

In [60]:
tensor

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[10,  7],
       [ 3,  4]], dtype=int32)>

## Matrix Multiplication with Tensors


Part 1

In ML, Matrix multiplication is one of the most common tensor operations.

In [61]:
tf.matmul(tensor, tensor)

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[121,  98],
       [ 42,  37]], dtype=int32)>

In [62]:
# if we do normal multiplication
tensor * tensor

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[100,  49],
       [  9,  16]], dtype=int32)>

In [63]:
tensor1 = tf.constant([[1,2,5],
                       [7,2,1],
                       [3,3,3]])
tensor2 = tf.constant([[3,5],
                       [6,7],
                       [1,8]])

tf.matmul(tensor1, tensor2)

<tf.Tensor: shape=(3, 2), dtype=int32, numpy=
array([[20, 59],
       [34, 57],
       [30, 60]], dtype=int32)>

In [64]:
# Matrix multiplication with Python operator "@"
tensor1 @ tensor2

<tf.Tensor: shape=(3, 2), dtype=int32, numpy=
array([[20, 59],
       [34, 57],
       [30, 60]], dtype=int32)>

📖 **Resources:** Info and examples of Matrix multiplication : https://www.mathsisfun.com/algebra/matrix-multiplying.html

**Rules:**
- The inner dimensions should be same/ must match (3,2) and (2,2)  
- The resulting matrix has the shape of the outer deminsions

In [65]:
X = tf.constant([[10,3],
                 [4,5],
                 [8,2]])

y = tf.constant([[10,3],
                 [4,5]])

tf.matmul(X,y)

<tf.Tensor: shape=(3, 2), dtype=int32, numpy=
array([[112,  45],
       [ 60,  37],
       [ 88,  34]], dtype=int32)>

In [66]:
X = tf.constant([[10,3],
                 [4,5],
                 [8,2]])

y = tf.constant([[10,3],
                 [4,5],
                 [8,2]])
tf.matmul(X,y)

InvalidArgumentError: {{function_node __wrapped__MatMul_device_/job:localhost/replica:0/task:0/device:CPU:0}} Matrix size-incompatible: In[0]: [3,2], In[1]: [3,2] [Op:MatMul] name: 

## Matrix Multiplication with Tensors - Part 2

In [67]:
y

<tf.Tensor: shape=(3, 2), dtype=int32, numpy=
array([[10,  3],
       [ 4,  5],
       [ 8,  2]], dtype=int32)>

In [68]:
# Lets change the shape of y
tf.reshape(y, shape=(2,3))

<tf.Tensor: shape=(2, 3), dtype=int32, numpy=
array([[10,  3,  4],
       [ 5,  8,  2]], dtype=int32)>

In [69]:
# try to multiply X by reshaped y
X @ tf.reshape(y, shape=(2,3))

<tf.Tensor: shape=(3, 3), dtype=int32, numpy=
array([[115,  54,  46],
       [ 65,  52,  26],
       [ 90,  40,  36]], dtype=int32)>

In [70]:
tf.matmul(tf.reshape(X, shape=(2,3)), y)

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[144,  53],
       [ 98,  59]], dtype=int32)>

In [71]:
tf.reshape(X, shape=(2,3)).shape, y.shape

(TensorShape([2, 3]), TensorShape([3, 2]))

In [72]:
# Difference between transpose and reshape

X, tf.transpose(X), tf.reshape(X, shape=(2,3))

(<tf.Tensor: shape=(3, 2), dtype=int32, numpy=
 array([[10,  3],
        [ 4,  5],
        [ 8,  2]], dtype=int32)>,
 <tf.Tensor: shape=(2, 3), dtype=int32, numpy=
 array([[10,  4,  8],
        [ 3,  5,  2]], dtype=int32)>,
 <tf.Tensor: shape=(2, 3), dtype=int32, numpy=
 array([[10,  3,  4],
        [ 5,  8,  2]], dtype=int32)>)

**Rule:** Always try Matrix multiplication with Transpose rather than reshape

In [73]:
tf.matmul(tf.transpose(X),y)

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[180,  66],
       [ 66,  38]], dtype=int32)>

## Matrix Multiplication with Tensors - Part 3

**The Dot Product**
Matrix multiplication is also referred as the dot product.

You can perform matrix multiplication using:
- `tf.matmul()`
- `tf.tensordor()`
- `@`

In [74]:
# Perform the dot product on X and y (required X or y to be transposed
tf.tensordot(X, tf.transpose(y), axes=1)

<tf.Tensor: shape=(3, 3), dtype=int32, numpy=
array([[109,  55,  86],
       [ 55,  41,  42],
       [ 86,  42,  68]], dtype=int32)>

In [75]:
tf.tensordot(tf.transpose(X),y,axes=1)

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[180,  66],
       [ 66,  38]], dtype=int32)>

**Note:**
You need to use **Transpose** rather than reshaping for **Matrix Multiplication**

## Changing the datatype of a tensor

In [76]:
# Create new tensor with default datatype (float32)

A = tf.constant([1.7, 8.3])
A.dtype

tf.float32

In [77]:
# Changing from float32 to float16
A = tf.cast(A, dtype=tf.float16)
A.dtype

tf.float16

In [78]:
B = tf.constant([1, 8])
B.dtype

tf.int32

In [79]:
# Changing from int32 to float32
B = tf.cast(B, dtype=tf.float32)
B.dtype

tf.float32

## Aggregating tensors

Aggregating tensors = Condensing them from multiple values down to a smaller amount of values.

In [87]:
# Get the absolute values
D = tf.constant([-7,-10])
D

<tf.Tensor: shape=(2,), dtype=int32, numpy=array([ -7, -10], dtype=int32)>

In [88]:
tf.abs(D)

<tf.Tensor: shape=(2,), dtype=int32, numpy=array([ 7, 10], dtype=int32)>

### Lets go through the following forms of aggregation:

* Get the minimum
* Get the maximum
* Get the mean of a tensor
* Get the sum of a tensor

In [91]:
# Create a random tenor with values between 0 and 100 of size 50
E = tf.constant(np.random.randint(0,100, size=50))
E

<tf.Tensor: shape=(50,), dtype=int64, numpy=
array([89, 74, 78, 36, 10, 47, 33, 54,  0, 97, 28, 43, 50,  8, 95, 94, 60,
       50, 46,  1, 47, 22,  3, 61, 12, 23, 97, 85, 95, 57, 58, 99, 71, 53,
       62,  0, 86, 48,  7, 34, 60, 11, 26, 55, 65, 32, 60, 92, 29, 91])>

In [94]:
E.shape, tf.size(E), E.ndim

(TensorShape([50]), <tf.Tensor: shape=(), dtype=int32, numpy=50>, 1)

In [95]:
# Find the minimum
tf.reduce_min(E)

<tf.Tensor: shape=(), dtype=int64, numpy=0>

In [96]:
# Find the Maximum
tf.reduce_max(E)

<tf.Tensor: shape=(), dtype=int64, numpy=99>

In [97]:
# Find the Mean
tf.reduce_mean(E)

<tf.Tensor: shape=(), dtype=int64, numpy=50>

In [98]:
# Find the Sum
tf.reduce_sum(E)

<tf.Tensor: shape=(), dtype=int64, numpy=2534>

In [103]:
# Find the Variance
tf.reduce_std(E)

AttributeError: module 'tensorflow' has no attribute 'reduce_std'

In [104]:
import tensorflow_probability as tfp
tfp.stats.variance(E)

<tf.Tensor: shape=(), dtype=int64, numpy=900>

In [107]:
tf.math.reduce_std(E)

TypeError: Input must be either real or complex. Received integer type <dtype: 'int64'>.

In [109]:
# Find the std deviation
tf.math.reduce_std(tf.cast(E, dtype=tf.float32))

<tf.Tensor: shape=(), dtype=float32, numpy=29.992292404174805>

In [110]:
tf.math.reduce_variance(tf.cast(E, dtype=tf.float32))

<tf.Tensor: shape=(), dtype=float32, numpy=899.5376586914062>

## Finding the positional maximum and minimum

- very import as Neural network has outputs of classification as probabilty and we need to show highest probability.

In [111]:
# Create a new tensor for finding positional minimum and maximum
tf.random.set_seed(42)
F = tf.random.uniform(shape=[50])
F

<tf.Tensor: shape=(50,), dtype=float32, numpy=
array([0.6645621 , 0.44100678, 0.3528825 , 0.46448255, 0.03366041,
       0.68467236, 0.74011743, 0.8724445 , 0.22632635, 0.22319686,
       0.3103881 , 0.7223358 , 0.13318717, 0.5480639 , 0.5746088 ,
       0.8996835 , 0.00946367, 0.5212307 , 0.6345445 , 0.1993283 ,
       0.72942245, 0.54583454, 0.10756552, 0.6767061 , 0.6602763 ,
       0.33695042, 0.60141766, 0.21062577, 0.8527372 , 0.44062173,
       0.9485276 , 0.23752594, 0.81179297, 0.5263394 , 0.494308  ,
       0.21612847, 0.8457197 , 0.8718841 , 0.3083862 , 0.6868038 ,
       0.23764038, 0.7817228 , 0.9671384 , 0.06870162, 0.79873943,
       0.66028714, 0.5871513 , 0.16461694, 0.7381023 , 0.32054043],
      dtype=float32)>

In [112]:
# Findind the positional maximum
tf.argmax(F)

<tf.Tensor: shape=(), dtype=int64, numpy=42>

In [113]:
F[tf.argmax(F)]

<tf.Tensor: shape=(), dtype=float32, numpy=0.967138409614563>

In [116]:
tf.reduce_max(F)

<tf.Tensor: shape=(), dtype=float32, numpy=0.967138409614563>

In [117]:
# Check for equality
F[tf.argmax(F)] ==tf.reduce_max(F)

<tf.Tensor: shape=(), dtype=bool, numpy=True>

In [114]:
tf.argmin(F)

<tf.Tensor: shape=(), dtype=int64, numpy=16>

In [115]:
F[tf.argmin(F)]

<tf.Tensor: shape=(), dtype=float32, numpy=0.009463667869567871>

## Squeezing A Tensor (removing all 1-D axes)

In [118]:
# Create a new tensor
tf.random.set_seed(42)
G = tf.constant(tf.random.uniform(shape=[50]), shape=(1,1,1,1,50))

In [119]:
G

<tf.Tensor: shape=(1, 1, 1, 1, 50), dtype=float32, numpy=
array([[[[[0.6645621 , 0.44100678, 0.3528825 , 0.46448255, 0.03366041,
           0.68467236, 0.74011743, 0.8724445 , 0.22632635, 0.22319686,
           0.3103881 , 0.7223358 , 0.13318717, 0.5480639 , 0.5746088 ,
           0.8996835 , 0.00946367, 0.5212307 , 0.6345445 , 0.1993283 ,
           0.72942245, 0.54583454, 0.10756552, 0.6767061 , 0.6602763 ,
           0.33695042, 0.60141766, 0.21062577, 0.8527372 , 0.44062173,
           0.9485276 , 0.23752594, 0.81179297, 0.5263394 , 0.494308  ,
           0.21612847, 0.8457197 , 0.8718841 , 0.3083862 , 0.6868038 ,
           0.23764038, 0.7817228 , 0.9671384 , 0.06870162, 0.79873943,
           0.66028714, 0.5871513 , 0.16461694, 0.7381023 , 0.32054043]]]]],
      dtype=float32)>

In [120]:
G.shape

TensorShape([1, 1, 1, 1, 50])

In [121]:
G_squeeze = tf.squeeze(G)
G_squeeze

<tf.Tensor: shape=(50,), dtype=float32, numpy=
array([0.6645621 , 0.44100678, 0.3528825 , 0.46448255, 0.03366041,
       0.68467236, 0.74011743, 0.8724445 , 0.22632635, 0.22319686,
       0.3103881 , 0.7223358 , 0.13318717, 0.5480639 , 0.5746088 ,
       0.8996835 , 0.00946367, 0.5212307 , 0.6345445 , 0.1993283 ,
       0.72942245, 0.54583454, 0.10756552, 0.6767061 , 0.6602763 ,
       0.33695042, 0.60141766, 0.21062577, 0.8527372 , 0.44062173,
       0.9485276 , 0.23752594, 0.81179297, 0.5263394 , 0.494308  ,
       0.21612847, 0.8457197 , 0.8718841 , 0.3083862 , 0.6868038 ,
       0.23764038, 0.7817228 , 0.9671384 , 0.06870162, 0.79873943,
       0.66028714, 0.5871513 , 0.16461694, 0.7381023 , 0.32054043],
      dtype=float32)>

In [122]:
G_squeeze.shape

TensorShape([50])

## One-Hot Encoding Tensors

In [124]:
# Create a list of indices
some_list = [0,1,2,3] # could be red ,green, blue, yellow

# One-Hot encode list of indices
tf.one_hot(some_list)

TypeError: Missing required positional argument

In [125]:
# Create a list of indices
some_list = [0,1,2,3] # could be red ,green, blue, yellow

# One-Hot encode list of indices
tf.one_hot(some_list, depth=4)

<tf.Tensor: shape=(4, 4), dtype=float32, numpy=
array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]], dtype=float32)>

In [126]:
# Specify custom values for one hot encoding
tf.one_hot(some_list, depth=4, on_value="Yo I love Deep Learning", off_value="I also like to Dance")

<tf.Tensor: shape=(4, 4), dtype=string, numpy=
array([[b'Yo I love Deep Learning', b'I also like to Dance',
        b'I also like to Dance', b'I also like to Dance'],
       [b'I also like to Dance', b'Yo I love Deep Learning',
        b'I also like to Dance', b'I also like to Dance'],
       [b'I also like to Dance', b'I also like to Dance',
        b'Yo I love Deep Learning', b'I also like to Dance'],
       [b'I also like to Dance', b'I also like to Dance',
        b'I also like to Dance', b'Yo I love Deep Learning']],
      dtype=object)>

In [127]:
# Specify custom values for one hot encoding
tf.one_hot(some_list, depth=4, on_value="DL", off_value="ML")

<tf.Tensor: shape=(4, 4), dtype=string, numpy=
array([[b'DL', b'ML', b'ML', b'ML'],
       [b'ML', b'DL', b'ML', b'ML'],
       [b'ML', b'ML', b'DL', b'ML'],
       [b'ML', b'ML', b'ML', b'DL']], dtype=object)>

In [128]:
tf.one_hot()

<tf.Tensor: shape=(4, 3), dtype=float32, numpy=
array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       [0., 0., 0.]], dtype=float32)>

## More maths operations
* `square`
* `square root`
* `log`

In [135]:
# Create a new tensor
H = tf.constant(range(1,9))
H

<tf.Tensor: shape=(8,), dtype=int32, numpy=array([1, 2, 3, 4, 5, 6, 7, 8], dtype=int32)>

In [136]:
# Square
tf.square(H)

<tf.Tensor: shape=(8,), dtype=int32, numpy=array([ 1,  4,  9, 16, 25, 36, 49, 64], dtype=int32)>

In [137]:
# Square root
tf.sqrt(H)

InvalidArgumentError: Value for attr 'T' of int32 is not in the list of allowed values: bfloat16, half, float, double, complex64, complex128
	; NodeDef: {{node Sqrt}}; Op<name=Sqrt; signature=x:T -> y:T; attr=T:type,allowed=[DT_BFLOAT16, DT_HALF, DT_FLOAT, DT_DOUBLE, DT_COMPLEX64, DT_COMPLEX128]> [Op:Sqrt] name: 

In [138]:
tf.sqrt(tf.cast(H, dtype=tf.float32))

<tf.Tensor: shape=(8,), dtype=float32, numpy=
array([1.       , 1.4142135, 1.7320508, 2.       , 2.236068 , 2.4494898,
       2.6457512, 2.828427 ], dtype=float32)>

In [141]:
#log
tf.math.log(tf.cast(H, dtype=tf.float32))

<tf.Tensor: shape=(8,), dtype=float32, numpy=
array([0.       , 0.6931472, 1.0986123, 1.3862944, 1.609438 , 1.7917595,
       1.9459102, 2.0794415], dtype=float32)>

## TensorFlow and Numpy compatibility

TensorFlow interacts beautifully with NumPy arrays.

In [142]:
# Creating a tensor directly from a NumPy array
J = tf.constant(np.array([3.,7.,10.]))
J

<tf.Tensor: shape=(3,), dtype=float64, numpy=array([ 3.,  7., 10.])>

In [143]:
# Converting Tensor back into Numpy array

np.array(J) , type(np.array(J))

(array([ 3.,  7., 10.]), numpy.ndarray)

In [144]:
J.numpy(), type(J.numpy())

(array([ 3.,  7., 10.]), numpy.ndarray)

In [145]:
# The default data types are different
numpy_J = tf.constant(np.array([3.,7.,10.]))
tensor_J = tf.constant([3.,7.,10.])

# Check datatypes of each
numpy_J.dtype, tensor_J.dtype

(tf.float64, tf.float32)